In [1]:
pip install transformers torch gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 96.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 80.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 87.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.2/62.2 MB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.9/321.9 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from transformers import pipeline
import spacy
import pandas as pd
import gradio as gr
from sklearn.linear_model import LinearRegression

import warnings
warnings.filterwarnings('ignore')

In [3]:
# Load pre-trained Hugging Face model for text classification
classifier = pipeline('zero-shot-classification', model="facebook/bart-large-mnli")

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cpu


In [4]:
def classify_prompt(prompt):
    labels = ["descriptive_statistics", "correlation", "regression", "others"]
    result = classifier(prompt, candidate_labels=labels)
    return result['labels'][0]  # Return the highest scored label

In [5]:
nlp = spacy.load("en_core_web_sm")

def extract_variables(prompt):
    doc = nlp(prompt)
    variables = [ent.text for ent in doc.ents if ent.label_ == "GPE" or ent.label_ == "ORG"]  # Customize for variables
    if len(variables) == 0:
        return None
    return variables

In [6]:
def load_data(file):
    if not file.name.endswith('.csv'):
        raise ValueError("Please upload a CSV file.")
    return pd.read_csv(file.name)  # Use file.name to read the file

def get_columns(df):
    return df.columns.tolist()

In [7]:
def descriptive_statistics(df, variables):
    return df[variables].describe()

def correlation(df, variables):
    return df[variables].corr()

def regression(df, dependent_var, independent_vars):
    X = df[independent_vars]
    y = df[dependent_var]
    model = LinearRegression()
    model.fit(X, y)
    return model.coef_, model.intercept_

In [8]:
def analyze_data(file, prompt):
    # Step 1: Classify the prompt
    analysis_type = classify_prompt(prompt)

    # Step 2: Extract variables from the prompt
    variables = extract_variables(prompt)
    if variables is None:
        return "Please specify the variables for analysis."

    # Step 3: Load the data and extract columns
    try:
        df = load_data(file)
    except ValueError as e:
        return str(e)

    columns = get_columns(df)

    if set(variables).issubset(columns):
        if analysis_type == "descriptive_statistics":
            result = descriptive_statistics(df, variables)
        elif analysis_type == "correlation":
            result = correlation(df, variables)
        elif analysis_type == "regression":
            result = regression(df, variables[0], variables[1:])
        else:
            result = "Analysis type not recognized."
    else:
        return "Some variables not found in the dataset."

    return result

In [10]:
# Create Gradio Interface
interface = gr.Interface(fn=analyze_data,
                         inputs=[gr.File(label="Upload CSV File"), gr.Textbox(label="Analysis Prompt")],
                         outputs="text",
                         title = "Gen AI Data Analysis Assistant")

interface.launch(debug = True)

Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f4bb543d4420d8ad89.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://f4bb543d4420d8ad89.gradio.live
